UN General Debate Transcripts Dataset from *Understanding State Preferences With Text As Data: Introducing the UN General Debate Corpus*

https://journals.sagepub.com/doi/epub/10.1177/2053168017712821

In this notebook, we generate temporal topic models of the UNGDC with various clustering and Mapper parameters.

Main packages: [temporal mapper](https://github.com/TutteInstitute/temporal-mapper) + [toponymy](https://github.com/TutteInstitute/toponymy).

The next cell contains all relevant parameters to be modified.


In [ ]:
"""
Parameters
"""
# Set this to agree with `UNGDC-DataPrep.ipynb`
data_path = "./output_data/UN-2026-03-18"

slice_numbers = [6,10,14]

## Mapper settings
import temporalmapper as tm
mapper_params = {
    "n_neighbors": 500,
    "slice_method": "data",
    "overlap":0.8,
    "kernel":tm.kernels.gaussian,
}
### THE CLUSTERER PARAMS DOMINATE RUNTIME -- MORE CLUSTERS IS EXPONENTIALLY SLOWER ###
clusterer_params = {
    'min_clusters':4,
    'max_layers':3,
    'base_min_cluster_size':100,
    'verbose':False
}

## Toponymy settings
compute_names = False

# Options: 'all-MiniLM-L6-v2' (fast, 384 dim), 'all-mpnet-base-v2' (better quality, 768 dim)
model_name = 'all-MiniLM-L6-v2'

toponymy_object_description = "excerpts from a speech"
toponymy_corpus_description = "United Nations General Debate Transcripts"

toponymy_exemplar_method = "central"
toponymy_keyphrase_method = "information_weighted"
toponymy_subtopic_method = "facility_location"

configuration_dict = {
    'slice_numbers':slice_numbers,
    'mapper_params':mapper_params,
    'model_name':model_name,
    'toponymy_object_description':toponymy_object_description,
    'toponymy_corpus_description':toponymy_corpus_description,
    'toponymy_exemplar_method':toponymy_exemplar_method,
    'toponymy_keyphrase_method':toponymy_keyphrase_method,
    'toponymy_subtopic_method':toponymy_subtopic_method,
    'clusterer_params':clusterer_params,
}

import json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
def print_ts(txt):
    now = datetime.now(ZoneInfo('US/Eastern'))
    formatted_datetime = now.strftime("%Y-%m-%d %H:%M:%S")
    print(formatted_datetime+": "+txt)

try:
    Path(data_path).mkdir(parents=True, exist_ok=True)
    # can't JSON serialize the kernel func so gotta do its name instead:
    kernel = mapper_params['kernel']
    configuration_dict['mapper_params']['kernel']=mapper_params['kernel'].__name__
    with open(data_path+'/configuration.json', 'w') as f:
        json.dump(configuration_dict, f, indent=4)
    configuration_dict['mapper_params']['kernel'] = kernel
    print_ts(f"Created experimental folder {data_path}, and saved configuation file.")
except Exception as e:
    print('Error:', e)

import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from tqdm.auto import tqdm
import pickle
# Register tqdm with pandas
tqdm.pandas()

# # Import the pre-processed dataset (see: UNGDC-DataPrep.ipynb)
# df = pd.read_parquet(
#     "https://huggingface.co/datasets/kalebr/un-general-debate-corpus-chunked/resolve/main/ungdc-all-chunked.parquet"
# )


# embedding_vectors = df['embedding'].to_numpy()
# reduced_vectors = df['reduced'].to_numpy()
# text = df['chunk_text'].to_numpy()
# time = df['year'].to_numpy()

# # Sort all arrays by time
# sorted_idx = np.argsort(time)
# time = time[sorted_idx]
# text = text[sorted_idx]
# reduced_vectors = np.vstack(reduced_vectors[sorted_idx])
# embedding_vectors = np.vstack(embedding_vectors[sorted_idx])
# reduced_vectors_with_time = np.hstack([reduced_vectors, time.reshape(-1,1)])

In [ ]:

## Toponymy Setup
from toponymy import Toponymy, ToponymyClusterer
from toponymy.cluster_layer import ClusterLayerText  
#from toponymy.llm_wrappers import AzureAINamer, AnthropicNamer
from toponymy.llm_wrappers import CohereNamer

api_key_file = "cohere.txt"
with open(api_key_file, 'r') as file:
    api_key = file.read().strip()


# Initialize with Cohere API
llm = CohereNamer(
    api_key=api_key,
    model="command-r-08-2024",  # Balanced performance and cost
    base_url="https://api.cohere.com"  # Optional custom endpoint
)

# Test connection  
print(llm.test_llm_connectivity())

from pathlib import Path
# if Path("topic_model.pkl").is_file():
#     print("Loading topic model.")
#     with open ('topic_model.pkl', 'rb') as f:
#         topic_model = pickle.load(f)
# else:
#     print("Computing topic model.")
#     topic_model = Toponymy(**toponymy_params)
#     topic_model.fit(text, embedding_vectors, reduced_vectors, **toponymy_fit_params)

In [ ]:

from sentence_transformers import SentenceTransformer
import torch
# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load the embedding model
model = SentenceTransformer(model_name, device=device)
print(f"Loaded model: {model_name}")

from mapperclusterer import MapperClusterer
base_clusterer = ToponymyClusterer(**clusterer_params)

toponymy_params = {
    'llm_wrapper':llm,
    'text_embedding_model':model,
    'object_description':"excerpts from a speech",
    'corpus_description':"United Nations General Debate Transcripts",
    'exemplar_delimiters':["<EXAMPLE_TRANSCRIPT>\n","\n</EXAMPLE_TRANSCRIPT>\n\n"],
}
toponymy_fit_params = {
    'exemplar_method':toponymy_exemplar_method,
    'keyphrase_method':toponymy_keyphrase_method,
    'subtopic_method':toponymy_subtopic_method,
}


In [ ]:
from toponymy.clustering import Clusterer, ClusterLayerText
from temporalmapper import TemporalMapper
from copy import deepcopy 
import networkx as nx
import numpy as np
from scipy.sparse import issparse
from sklearn.utils.validation import check_is_fitted, check_array
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

print("Checking cluster sizes:")
for n_slices in {10}:
    for density_based in {False,True}:
        mapper_params['n_slices'] = n_slices
        mapper_params['density_based'] = density_based
        clusterer_params['base_min_cluster_size'] = int(120/np.sqrt(n_slices))
        print(f"Computing for N={n_slices}, db={density_based}")
        base_clusterer = ToponymyClusterer(**clusterer_params)
        try:
            clusterer = MapperClusterer(
                base_clusterer=base_clusterer,
                mapper_params=mapper_params,
                verbose=True,
            )
            clusterer.fit(reduced_vectors_with_time, embedding_vectors)
        except Exception as e:
            print(f"Failed to fit db={density_based} slices={n_slices}", e)
            raise e

In [ ]:
from toponymy.serialization import TopicModel
import pickle 

dp = Path(data_path+"/saved")
dp.mkdir(parents=True, exist_ok=True)
print("Computing...")
for n_slices in slice_numbers:
    for density_based in {False,True}:
        fp = Path(data_path+f"/saved/tm_{n_slices}_{density_based}.tm.zip")
        if fp.is_file() and (fp.stat().st_size>1024):
            print(f"Skipping {n_slices}, db={density_based}")
            continue
        mapper_params['n_slices'] = n_slices
        mapper_params['density_based'] = density_based
        clusterer_params['base_min_cluster_size'] = int(120/np.sqrt(n_slices))
        print(f"Computing for N={n_slices}, db={density_based}")
        base_clusterer = ToponymyClusterer(**clusterer_params)
        try:
            clusterer = MapperClusterer(
                base_clusterer=base_clusterer,
                mapper_params=mapper_params
            )
            clusterer.fit(reduced_vectors_with_time, embedding_vectors)
            toponymy_params['clusterer'] = clusterer
            toponymy = Toponymy(**toponymy_params)
            toponymy.fit(
                text,
                embedding_vectors,
                reduced_vectors_with_time,
            )
            tm = TopicModel.from_toponymy(toponymy)
            tm.to_file(str(fp))
            with open(data_path+f"/saved/tm_clusterer_{n_slices}_{density_based}.pkl", 'wb') as f:
                pickle.dump(toponymy.clusterer, f)
        except Exception as e:
            print(f"Failed to fit db={density_based} slices={n_slices}", e)
            raise e

## Plotting

In [ ]:
from temporalmapper.plotting import squarify_text
import matplotlib.pyplot as plt
from plotly.graph_objects import Layout

def make_cluster_labels(clusterer, topic_model, hide_deg2=True):
    cluster_labels = {
        l:{} for l in range(len(clusterer.mappers_))
    }
    
    for layer, mapper in enumerate(clusterer.mappers_):
        for node in mapper.graph.nodes():
            if (hide_deg2 and (mapper.graph.in_degree(node) == 1)
            and (mapper.graph.out_degree(node)==1) and (layer!=2)):
                 cluster_labels[layer][node] = ""
            else:
                cluster = clusterer.topic_map_[layer][node]
                topic=topic_model.topic_df.query(f"layer=={layer} and cluster=={cluster}")
                if len(topic)==0:
                    cluster_labels[layer][node]=""
                else:
                    cluster_labels[layer][node] = squarify_text(
                        topic.name.values[0]
                    )
    return cluster_labels

def temporal_plot(mapper, topic_model, layer, cluster_labels, vertices=None):
    fig, ax = plt.subplots(1,1, figsize=(16,10))
    if vertices is None:
        vertices = mapper.graph.nodes()
    
    fontsize = {2:8,1:6,0:5}
    if layer == 0:
        layout = 'barycenter'
    else:
        layout = 'ordered'
    mapper.temporal_plot(
        ax=ax,
        cluster_labels = cluster_labels[layer],
        layout = layout,
        cluster_label_kwargs = dict(fontsize=fontsize[layer]),
        edge_weight_bounds = (1,5),
        node_size_bounds = (5,50),
        node_scaling = 5,
        vertices = vertices,
    )
    label_times = mapper.midpoints
    label_text = [int(label) for label in label_times]
    ax.set_xticks(label_times, labels=label_text)
    ax.tick_params(axis='x', labelrotation=90)
    ax.tick_params(bottom=True, labelbottom=True)
    N = mapper._mapper.n_slices
    g = mapper._mapper.overlap
    k = mapper._mapper.n_neighbors
    title_str = f"UNGDC Mapper Temporal Embedding\n Parameters: N={N}, g={g}, k={k}, Layer={layer} "
    if mapper.density_based:
        title_str += "(density based)"
    else:
        title_str += "(standard mapper)"
    ax.set_title(title_str,  fontsize=18)
    plt.tight_layout()
    return fig

def interactive_plot(mapper, topic_model, layer, cluster_labels, vertices=None):
    if vertices is None:
        vertices = mapper.graph.nodes()
    
    fontsize = {2:8,1:6,0:5}
    layout = 'ordered'

    label_times = mapper.midpoints
    label_text = [int(label) for label in label_times]
    N = mapper._mapper.n_slices
    g = mapper._mapper.overlap
    k = mapper._mapper.n_neighbors
    title_str = f"UNGDC Mapper Temporal Embedding, Layer={layer}, "
    if mapper.density_based:
        title_str += "(density based)"
    else:
        title_str += "(standard mapper)"

    title_str += f" | Params: N={N}, g={g}, k={k}"

    graph_layout = Layout(
        title=dict(
            text=title_str
        ),
        width=1000,
        height=600,
        showlegend=False,
    )

    fig = mapper.interactive_temporal_plot(
        hover_text = cluster_labels[layer],
        layout = layout,
        graph_layout=graph_layout,
        edge_weight_bounds = (1,5),
        node_size_bounds = (5,50),
        node_scaling = 5,
        vertices = vertices,
    )

    return fig

In [ ]:
# Plotting cell.
import gc
from toponymy.serialization import TopicModel
from pathlib import Path

image_path = Path(data_path+"/images")
image_path.mkdir(parents=True, exist_ok=True)

directory_path = Path(data_path+"/saved") 
pkl_files = list(directory_path.glob("*.pkl"))

for file_path in pkl_files:
    print('plotting from', file_path)
    if "False" in str(file_path):
        print("passing ", file_path)
        continue
    if "clusterer_6" in str(file_path):
        print("passing ", file_path)
        continue
    with open(file_path, 'rb') as f:
        clusterer = pickle.load(f)
        
    n_slices = clusterer.mapper_params['n_slices']
    density_based = clusterer.mapper_params['density_based']
    topic_model = TopicModel.from_file(data_path+f"/saved/tm_{n_slices}_{density_based}.tm.zip")
    cluster_labels = make_cluster_labels(clusterer, topic_model)
    for layer in range(len(clusterer.mappers_)):
        fig = temporal_plot(clusterer.mappers_[layer], topic_model, layer, cluster_labels)
        fig.savefig(str(image_path) + "/" + f'tp_N{n_slices}_db{density_based}_layer{layer}.png')
        fig.show()
        plt.close(fig)
        del fig
    # doing manual GC because this crashes on my 16GB ram PC otherwise.
    del clusterer, topic_model, cluster_labels
    gc.collect()


In [ ]:
def connected_component(clusterer, vertex):
    """ Returns nodes in the connected component of `vertex`"""
    layer, time, cluster = vertex.split(":")
    layer = int(layer)
    mapper = clusterer.mappers_[layer]
    subgraph = mapper.vertex_subgraph(time+":"+cluster)
    return subgraph

def down_one_level(clusterer, cluster_labels, vertex):
    """ Returns the nodes connected below 'vertex' in the hierarchy """
    layer, time, cluster = vertex.split(":")
    layer = int(layer)
    reverse_topic_map={}
    for l in range(len(clusterer.topic_map_)):
        reverse_topic_map[l] = {
            num:[] for num in clusterer.topic_map_[l].values()
        }
        for node, num in clusterer.topic_map_[l].items():
            reverse_topic_map[l][num].append(node)
    
    subgraphs = {l:[] for l in range(layer+1)}
    for vert in connected_component(clusterer, vertex):
        time, premap_cluster = vert.split(":")
        cluster = clusterer.topic_map_[layer][f"{time}:{premap_cluster}"]
        subgraphs[layer].append(vert)
        for subvertex in clusterer.cluster_tree_[(layer, cluster)]:
            sublayer, subcluster = subvertex
            for x in reverse_topic_map[sublayer][subcluster]: 
                for y in connected_component(clusterer, f"{sublayer}:{x}"):
                    subgraphs[sublayer].append(y)
    subgraphs = {l:np.unique(subgraphs[l]) for l in subgraphs.keys()}
    return subgraphs

In [ ]:
for file_path in pkl_files:
    print(file_path)

In [ ]:
with open('output_data/UN-2026-03-18/saved/tm_clusterer_10_True.pkl', 'rb') as f:
        clusterer = pickle.load(f)    
n_slices = clusterer.mapper_params['n_slices']
density_based = clusterer.mapper_params['density_based']
topic_model = TopicModel.from_file(data_path+f"/saved/tm_{n_slices}_{density_based}.tm.zip")
cluster_labels = make_cluster_labels(clusterer, topic_model, hide_deg2=False)

## Modify ====
image_path = Path(data_path+"/images")
layer = 1
t=4
l=2
c=1
## ===========
cluster = clusterer.topic_map_[l][f"{t}:{c}"]
topic = topic_model.topic_df.query(f"layer=={l} and cluster=={cluster}")
name = topic.name.values[0]
print("Mapper graph under", name)
subgraphs = down_one_level(clusterer, cluster_labels, f"{l}:{t}:{c}")

fig = temporal_plot(
    clusterer.mappers_[layer],
    topic_model,
    layer,
    cluster_labels,
    vertices=subgraphs[layer],
)
fig.savefig(str(image_path) + "/" + f'subgraph_{name}_layer{layer}.png')
fig.show()

In [ ]:
## Modify ====
image_path = Path(data_path+"/images")
layer = 1
t=3
l=1
c=2
## ===========
cluster = clusterer.topic_map_[l][f"{t}:{c}"]
topic = topic_model.topic_df.query(f"layer=={l} and cluster=={cluster}")
name = topic.name.values[0]
print("Mapper Subgraph of", name)
subgraph = connected_component(clusterer, f"{l}:{t}:{c}")

fig = temporal_plot(
    clusterer.mappers_[layer],
    topic_model,
    layer,
    cluster_labels,
    vertices=subgraph,
)
fig.savefig(str(image_path) + "/" + f'subgraph_{name}_layer{layer}.png')
fig.show()